In [3]:
import numpy as np
import pandas as pd

In [4]:
# 1. Create three DataFrames with different time frequencies (e.g., 5-min, 10-min, 15-min).
# 1. 5-minute frequency DataFrame
index_5m = pd.date_range("2026-07-15 08:00:00", periods=5, freq="5min")
df_5m = pd.DataFrame({"Value": [10, 15, 12, 18, 14]}, index=index_5m)

# 2. 10-minute frequency DataFrame
index_10m = pd.date_range("2026-07-15 08:00:00", periods=5, freq="10min")
df_10m = pd.DataFrame({"Value": [100, 150, 120, 180, 140]}, index=index_10m)

# 3. 15-minute frequency DataFrame
index_15m = pd.date_range("2026-07-15 08:00:00", periods=5, freq="15min")
df_15m = pd.DataFrame({"Value": [1.0, 1.5, 1.2, 1.8, 1.4]}, index=index_15m)

print("5-Min DataFrame:\n", df_5m, "\n")
print("10-Min DataFrame:\n", df_10m, "\n")
print("15-Min DataFrame:\n", df_15m)

5-Min DataFrame:
                      Value
2026-07-15 08:00:00     10
2026-07-15 08:05:00     15
2026-07-15 08:10:00     12
2026-07-15 08:15:00     18
2026-07-15 08:20:00     14 

10-Min DataFrame:
                      Value
2026-07-15 08:00:00    100
2026-07-15 08:10:00    150
2026-07-15 08:20:00    120
2026-07-15 08:30:00    180
2026-07-15 08:40:00    140 

15-Min DataFrame:
                      Value
2026-07-15 08:00:00    1.0
2026-07-15 08:15:00    1.5
2026-07-15 08:30:00    1.2
2026-07-15 08:45:00    1.8
2026-07-15 09:00:00    1.4


In [5]:
# 2. Merge them using outer join to preserve all timestamps .
# Outer join merge using concat
# axis=1 aligns rows by their index labels
merged_df = pd.concat([df_5m, df_10m, df_15m], axis=1, join="outer")

print(merged_df)

                     Value  Value  Value
2026-07-15 08:00:00   10.0  100.0    1.0
2026-07-15 08:05:00   15.0    NaN    NaN
2026-07-15 08:10:00   12.0  150.0    NaN
2026-07-15 08:15:00   18.0    NaN    1.5
2026-07-15 08:20:00   14.0  120.0    NaN
2026-07-15 08:30:00    NaN  180.0    1.2
2026-07-15 08:40:00    NaN  140.0    NaN
2026-07-15 08:45:00    NaN    NaN    1.8
2026-07-15 09:00:00    NaN    NaN    1.4


In [6]:
# 3. Use fillna(method='ffill') to forward-fill missing values.
# Apply forward fill to propagate the last valid observation forward
filled_df = merged_df.ffill()

print(filled_df)

                     Value  Value  Value
2026-07-15 08:00:00   10.0  100.0    1.0
2026-07-15 08:05:00   15.0  100.0    1.0
2026-07-15 08:10:00   12.0  150.0    1.0
2026-07-15 08:15:00   18.0  150.0    1.5
2026-07-15 08:20:00   14.0  120.0    1.5
2026-07-15 08:30:00   14.0  180.0    1.2
2026-07-15 08:40:00   14.0  140.0    1.2
2026-07-15 08:45:00   14.0  140.0    1.8
2026-07-15 09:00:00   14.0  140.0    1.4


In [9]:
# 4. Convert a wide DataFrame (countries as columns, years as rows) to long format.

# Completely replace your previous cell content with this:
data = {
    "Year": [2021, 2022, 2023],
    "USA": [100, 105, 110],
    "Canada": [80, 85, 88],
    "Germany": [90, 92, 95],
}

wide_df = pd.DataFrame(data)

long_df = wide_df.melt(
    id_vars=["Year"],
    value_vars=["USA", "Canada", "Germany"],
    var_name="Country",
    value_name="Metric",
)

print(long_df)

   Year  Country  Metric
0  2021      USA     100
1  2022      USA     105
2  2023      USA     110
3  2021   Canada      80
4  2022   Canada      85
5  2023   Canada      88
6  2021  Germany      90
7  2022  Germany      92
8  2023  Germany      95


In [11]:
# 5. Challenge: Use pd.merge_asof to merge stock prices (irregular) with quarterly earnings reports (regular) . 

# 1. Create irregular stock price data (must be sorted by timestamp)
stock_data = {
    "Timestamp": pd.to_datetime(
        [
            "2026-01-10 10:00:00",
            "2026-03-15 14:30:00",
            "2026-04-02 09:15:00",
            "2026-06-20 16:00:00",
            "2026-07-05 11:00:00",
        ]
    ),
    "Price": [150.50, 152.20, 158.00, 156.40, 162.10],
}
df_stocks = pd.DataFrame(stock_data).sort_values("Timestamp")

# 2. Create regular quarterly earnings data (must be sorted by date)
earnings_data = {
    "Report_Date": pd.to_datetime(["2026-01-01", "2026-04-01", "2026-07-01"]),
    "EPS": [1.25, 1.40, 1.35],
}
df_earnings = pd.DataFrame(earnings_data).sort_values("Report_Date")

# 3. Perform the asof merge
merged_df = pd.merge_asof(
    df_stocks,
    df_earnings,
    left_on="Timestamp",
    right_on="Report_Date",
    direction="backward",  # Aligns stock price with the most recent past earnings report
)

print(merged_df)


            Timestamp  Price Report_Date   EPS
0 2026-01-10 10:00:00  150.5  2026-01-01  1.25
1 2026-03-15 14:30:00  152.2  2026-01-01  1.25
2 2026-04-02 09:15:00  158.0  2026-04-01  1.40
3 2026-06-20 16:00:00  156.4  2026-04-01  1.40
4 2026-07-05 11:00:00  162.1  2026-07-01  1.35
